Topic: PySpark GroupBy Aggregation

Objective:
This script demonstrates how to group and aggregate sales/order data using PySpark.

Workflow:
1. Read raw orders CSV data from a Databricks Volume
2. Filter only completed orders
3. Group data by product category
4. Calculate total orders, total revenue, average order amount, minimum order amount, and maximum order amount
5. Write the aggregated result as Parquet

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, min, max, round


spark = SparkSession.builder.appName("dayily-practice").getOrCreate()


INPUT_PATH = "/Volumes/workspace/data/daily_practice/data/orders.csv"
OUTPUT_PATH = "/Volumes/workspace/data/daily_practice/output/category_sales_summary"


raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)


print("Raw Orders Data")
display(raw_df)

print("Raw Schema")
raw_df.printSchema()

print(f"Raw Record Count: {raw_df.count()}")




Raw Orders Data


order_id,customer_id,product_category,order_amount,order_status
1001,1,Electronics,1200.5,Completed
1002,2,Furniture,750.0,Completed
1003,3,Electronics,300.0,Cancelled
1004,1,Stationery,45.75,Completed
1005,4,Furniture,600.25,Completed
1006,2,Electronics,999.99,Completed
1007,5,Stationery,30.0,Pending
1008,3,Furniture,450.0,Completed
1009,1,Electronics,1500.0,Completed
1010,4,Stationery,80.0,Completed


Raw Schema
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- order_amount: double (nullable = true)
 |-- order_status: string (nullable = true)

Raw Record Count: 10


In [0]:
completed_orders_df = (
    raw_df
    .filter(col("order_status") == "Completed")
    .filter(col("order_amount").isNotNull())
)


category_summary_df = (
    completed_orders_df
    .groupBy("product_category")
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("order_amount"), 2).alias("total_revenue"),
        round(avg("order_amount"), 2).alias("average_order_amount"),
        round(min("order_amount"), 2).alias("minimum_order_amount"),
        round(max("order_amount"), 2).alias("maximum_order_amount")
    )
    .orderBy(col("total_revenue").desc())
)


print("Category Sales Summary")
display(category_summary_df)


(
    category_summary_df.write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)


print(f"Aggregated data written successfully to: {OUTPUT_PATH}")

Category Sales Summary


product_category,total_orders,total_revenue,average_order_amount,minimum_order_amount,maximum_order_amount
Electronics,3,3700.49,1233.5,999.99,1500.0
Furniture,3,1800.25,600.08,450.0,750.0
Stationery,2,125.75,62.88,45.75,80.0


Aggregated data written successfully to: /Volumes/workspace/data/daily_practice/output/category_sales_summary
